# How small can you go?

Every aggregation trades **size** for **accuracy**. You have two levers — the
number of typical **periods** and the number of **segments** per period (see
[Segmentation](segmentation.ipynb)) — and this page is about choosing them: first
by hand, then by letting tsam search the whole space for you.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## The trade-off, by hand

More time steps means more accuracy but less compression. Sweeping the number of
periods shows the classic diminishing-returns curve — error drops fast, then
flattens. The "knee" is where extra periods stop paying off:

In [ ]:
rows = []
for k in [2, 4, 6, 8, 12, 16, 24]:
    r = tsam.aggregate(data, n_clusters=k, period_duration="1D")
    rows.append(
        {
            "n_clusters": k,
            "timesteps": k * r.n_timesteps_per_period,
            "rmse": float(r.accuracy.rmse.mean()),
        }
    )
sweep = pd.DataFrame(rows)
px.line(
    sweep,
    x="timesteps",
    y="rmse",
    markers=True,
    title="Accuracy vs. size — number of periods",
)

## Both levers at once: a target reduction

Periods and segments interact, so searching by hand gets tedious fast. Give
[`find_optimal_combination`](../api/tuning.md) a target **data reduction** and it
searches period/segment combinations, returning the most accurate one that hits
your budget:

In [ ]:
from tsam.tuning import find_optimal_combination

best = find_optimal_combination(
    data,
    data_reduction=0.1,
    period_duration="1D",  # keep ~10% of the time steps
    n_jobs=1,
    show_progress=False,
)
print(f"best within budget: {best.n_clusters} periods x {best.n_segments} segments")
print(f"  time steps: {best.n_clusters * best.n_segments}  |  RMSE: {best.rmse:.4f}")

## Map the whole frontier

To choose deliberately instead of against one fixed budget,
[`find_pareto_front`](../api/tuning.md) finds the best aggregation at each size —
the accuracy-vs-size frontier:

In [ ]:
from tsam.tuning import find_pareto_front

pareto = find_pareto_front(
    data,
    period_duration="1D",
    timesteps=range(12, 96, 12),
    n_jobs=1,
    show_progress=False,
)
pareto.summary

In [ ]:
pareto.plot()

Each point mixes periods and segments differently — the table shows tsam often
prefers more periods with fewer segments, or vice versa, depending on the budget.

## Pull a result off the frontier

The frontier is not just a picture: look up the best aggregation for a given size
(or accuracy) and use it directly. `find_by_timesteps` / `find_by_rmse` return an
ordinary `AggregationResult`:

In [ ]:
small = pareto.find_by_timesteps(36)  # tightest budget on the curve
print(
    f"at 36 steps: {small.n_clusters} periods x {small.n_segments} segments, "
    f"RMSE {small.accuracy.rmse.mean():.4f}"
)

## Coarse vs. fine, side by side

A picture of what the budget buys: compare a coarse and a fine point from the
frontier against the original load over one week. The fine one tracks the detail;
the coarse one keeps only the broad shape:

In [ ]:
fine = pareto.find_by_timesteps(84)
week = slice("2010-01-11", "2010-01-17")
comparison = pd.DataFrame(
    {
        "original": data.loc[week, "Load"],
        f"coarse ({small.n_clusters}x{small.n_segments})": small.reconstructed.loc[
            week, "Load"
        ],
        f"fine ({fine.n_clusters}x{fine.n_segments})": fine.reconstructed.loc[
            week, "Load"
        ],
    }
)
px.line(comparison, title="One week: coarse vs. fine aggregation")

Any point on the frontier is a ready-to-use `AggregationResult` — hand it straight
to your model. See the [Optimization workflow](optimization_workflow.ipynb).